<a href="https://colab.research.google.com/github/gunasekhara67-del/PROJECTS/blob/main/AI_Business_Strategy_%26_Executive_Decision_Intelligence_Platform.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
!pip install -q pandas numpy matplotlib seaborn google-genai gradio

In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import gradio as gr
from getpass import getpass
from google import genai

In [ ]:
GEMINI_API_KEY = getpass("Enter YOUR API KEY: ")
client = genai.Client(api_key=GEMINI_API_KEY)

In [14]:
data={
    "Month":["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug"],
    "Sales":[82000,87000,91000,76000,99000,108000,112000,118000],
    "Orders":[820,850,900,770,970,1040,1080,1120],
    "Customers":[620,650,670,610,720,760,790,830],
    "Returns":[34,34,40,52,41,41,40,39],
    "Support_Tickets":[110,115,108,140,120,112,105,101],
    "Marketing_Spend":[12000,12500,13000,15000,14500,15000,15200,16000]
}
df=pd.DataFrame(data)
df["Return_Rate"]=df["Returns"]/df["Orders"]*100
df["Average_Order_Value"]=df["Sales"]/df["Orders"]
sample_path="/content/project3_business_data.csv"
df.to_csv(sample_path,index=False)
print("Sample CSV:",sample_path)
df.head()

Sample CSV: /content/project3_business_data.csv


,Month,Sales,Orders,Customers,Returns,Support_Tickets,Marketing_Spend,Return_Rate,Average_Order_Value
0,Jan,82000,820,620,34,110,12000,4.146341,100.000000
1,Feb,87000,850,650,34,115,12500,4.000000,102.352941
2,Mar,91000,900,670,40,108,13000,4.444444,101.111111
3,Apr,76000,770,610,52,140,15000,6.753247,98.701299
4,May,99000,970,720,41,120,14500,4.226804,102.061856


In [16]:
feedback=pd.DataFrame({
    "Feedback":[
        "Very good service and fast delivery","Product quality was excellent",
        "Delivery was okay","The product arrived late","Customer support was helpful",
        "I am unhappy with the return process","Great shopping experience",
        "Delivery delay was frustrating","Good price and quality","Support response was slow"
    ]
})
feedback_path="/content/project3_feedback.csv"
feedback.to_csv(feedback_path,index=False)
print("Feedback CSV:",feedback_path)
feedback.head()

Feedback CSV: /content/project3_feedback.csv


,Feedback
0,Very good service and fast delivery
1,Product quality was excellent
2,Delivery was okay
3,The product arrived late
4,Customer support was helpful


In [17]:
def analyze_business(file_path):
    df=pd.read_csv(file_path)

    if "Sales" not in df.columns or "Orders" not in df.columns:
        raise ValueError("Business CSV must contain Sales and Orders columns.")

    df["Average_Order_Value"]=df["Sales"]/df["Orders"]
    df["Sales_Growth_%"]=df["Sales"].pct_change()*100
    df["Growth_Risk"]=np.where(df["Sales_Growth_%"]<0,"Risk","Normal")

    if "Returns" in df.columns:
        df["Return_Rate"]=df["Returns"]/df["Orders"]*100
        df["Return_Risk"]=np.where(df["Return_Rate"]>5,"High","Normal")

    report={
        "Total Sales":float(df["Sales"].sum()),
        "Average Monthly Sales":float(df["Sales"].mean()),
        "Total Orders":int(df["Orders"].sum()),
        "Average Order Value":float(df["Average_Order_Value"].mean()),
        "Latest Sales":float(df["Sales"].iloc[-1]),
        "Latest Sales Growth %":float(df["Sales_Growth_%"].iloc[-1]) if len(df)>1 else 0
    }

    if "Customers" in df.columns:
        report["Latest Customers"]=int(df["Customers"].iloc[-1])
    if "Return_Rate" in df.columns:
        report["Average Return Rate"]=float(df["Return_Rate"].mean())
        if "Month" in df.columns:
            report["High Return Risk Periods"]=df.loc[df["Return_Rate"]>5,"Month"].tolist()

    return df,report

In [18]:
def analyze_feedback(file_path):
    feedback=pd.read_csv(file_path)
    if "Feedback" not in feedback.columns:
        raise ValueError("Feedback CSV must contain a Feedback column.")

    positive_words=["good","great","excellent","helpful","fast","quality"]
    negative_words=["late","unhappy","delay","frustrating","slow"]

    def sentiment(text):
        text=str(text).lower()
        positive=sum(w in text for w in positive_words)
        negative=sum(w in text for w in negative_words)
        if positive>negative:
            return "Positive"
        if negative>positive:
            return "Negative"
        return "Neutral"

    feedback["Sentiment"]=feedback["Feedback"].apply(sentiment)
    return feedback,feedback["Sentiment"].value_counts().to_dict()

In [19]:
def create_business_charts(df,feedback):
    charts=[]
    x=df["Month"] if "Month" in df.columns else df.index

    plt.figure(figsize=(8,4))
    sns.lineplot(x=x,y=df["Sales"],marker="o")
    plt.title("Monthly Sales Trend")
    plt.tight_layout()
    p="/content/project3_sales_trend.png"
    plt.savefig(p,dpi=150)
    plt.close()
    charts.append(p)

    if "Return_Rate" in df.columns:
        plt.figure(figsize=(8,4))
        sns.lineplot(x=x,y=df["Return_Rate"],marker="o")
        plt.title("Return Rate Trend")
        plt.tight_layout()
        p="/content/project3_return_rate.png"
        plt.savefig(p,dpi=150)
        plt.close()
        charts.append(p)

    counts=feedback["Sentiment"].value_counts()
    plt.figure(figsize=(7,4))
    sns.barplot(x=counts.index,y=counts.values)
    plt.title("Customer Feedback Sentiment")
    plt.tight_layout()
    p="/content/project3_sentiment.png"
    plt.savefig(p,dpi=150)
    plt.close()
    charts.append(p)

    if "Return_Risk" in df.columns:
        plt.figure(figsize=(6,4))
        df["Return_Risk"].value_counts().plot.pie(autopct="%1.0f%%",ylabel="")
        plt.title("Return Risk Distribution")
        plt.tight_layout()
        p="/content/project3_risk_pie.png"
        plt.savefig(p,dpi=150)
        plt.close()
        charts.append(p)
    return charts

In [20]:
def generate_executive_summary(report,feedback_counts):
    prompt=f'''
You are an Executive Decision Intelligence Assistant.
Use only the supplied analytics.

Create:
1. Executive summary
2. Important trends
3. Risks
4. Opportunities
5. Recommended actions
6. Monitoring points

Do not invent metrics. Clearly distinguish analytics from recommendations.

Business analytics:
{report}

Feedback sentiment:
{feedback_counts}
'''
    response=client.models.generate_content(
        model="gemini-3.5-flash",
        contents=prompt
    )
    return response.text

In [21]:
def scenario_analysis(df):
    latest=df.iloc[-1]
    current_sales=float(latest["Sales"])
    scenario_sales=current_sales*1.10

    if "Return_Rate" in df.columns:
        current_return=float(latest["Return_Rate"])
        scenario_return=max(0,current_return-1)
    else:
        current_return=None
        scenario_return=None

    text=f"Current sales: {current_sales:,.2f}\nScenario sales (+10%): {scenario_sales:,.2f}"
    if current_return is not None:
        text+=f"\nCurrent return rate: {current_return:.2f}%\nScenario return rate (-1 point): {scenario_return:.2f}%"
    return text

In [22]:
def gradio_project3(business_file,feedback_file):
    if business_file is None:
        return pd.DataFrame(),pd.DataFrame(),pd.DataFrame(),"Upload a business CSV.",[]

    df,report=analyze_business(business_file)

    if feedback_file is not None:
        feedback,feedback_counts=analyze_feedback(feedback_file)
    else:
        feedback=pd.DataFrame(columns=["Feedback","Sentiment"])
        feedback_counts={}

    charts=create_business_charts(df,feedback)
    ai_text=generate_executive_summary(report,feedback_counts)
    scenario=scenario_analysis(df)

    return pd.DataFrame([report]),df,feedback,ai_text+"\n\n### Scenario Analysis\n"+scenario,charts

with gr.Blocks(title="AI Business Strategy & Executive Decision Intelligence") as app:
    gr.Markdown("# AI Business Strategy & Executive Decision Intelligence Platform")
    gr.Markdown("Upload business data and customer feedback to generate KPIs, trends, risks, opportunities, charts and Gemini-assisted executive insights.")

    business_file=gr.File(label="Upload Business CSV",file_types=[".csv"],type="filepath")
    feedback_file=gr.File(label="Upload Customer Feedback CSV",file_types=[".csv"],type="filepath")
    analyze_button=gr.Button("Generate Executive Intelligence")

    kpi_output=gr.Dataframe(label="Business KPI Summary")
    business_output=gr.Dataframe(label="Analyzed Business Data")
    feedback_output=gr.Dataframe(label="Customer Feedback + Sentiment")
    ai_output=gr.Markdown(label="Gemini Executive Intelligence")
    chart_output=gr.Gallery(label="Charts",columns=2,height="auto")

    analyze_button.click(
        gradio_project3,
        inputs=[business_file,feedback_file],
        outputs=[kpi_output,business_output,feedback_output,ai_output,chart_output]
    )

app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d7413597df16029edd.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
